# Notebook 02: Concurrencia, Asincronía y asyncio

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/fdd_p26/blob/main/clase/16_computo/code/02_concurrencia_asyncio.ipynb)

**Módulo 16 — Clase 2**

Este notebook acompaña los archivos `03_concurrencia_y_asincronia.md` y `04a_asyncio_fundamentos.md`.

Secciones **** se trabajan durante la sesión.  
Secciones **** se completan después.

---

In [2]:
import asyncio
import time
import threading
import os
import sys

print(f'Python {sys.version}')
print(f'asyncio version: {asyncio.__version__ if hasattr(asyncio, "__version__") else "built-in"}')

# Jupyter ya tiene un event loop corriendo — podemos usar await directamente en las celdas
# Si usas un script .py, necesitas asyncio.run(main())

Python 3.14.3 (main, Feb 17 2026, 19:22:15) [GCC 13.3.0]
asyncio version: built-in


## Sección 1: await secuencial vs asyncio.gather — la diferencia central

La diferencia entre M2 (await secuencial) y M4 (gather) es una línea de código.
Medir los tiempos hace la diferencia completamente visible.

In [8]:
# Tarea simulada con I/O-bound: espera τ segundos
async def tarea_io(nombre: str, duracion: float) -> str:
    # exec(τᵢ): inicializar
    inicio = time.perf_counter()
    # wait(τᵢ): simula I/O (llamada a API, lectura de BD, etc.)
    await asyncio.sleep(duracion)
    # exec(τᵢ): procesar resultado
    elapsed = time.perf_counter() - inicio
    return f'{nombre}: {elapsed:.2f}s'

DURACION = 1.0  # cada tarea tarda 1s de I/O
N_TAREAS = 5

# --- M2: await secuencial (esperas NO explotadas) ---
t0 = time.perf_counter()
resultados_m2 = []
for i in range(N_TAREAS):
    r = await tarea_io(f'τ{i+1}', DURACION)
    resultados_m2.append(r)
t_m2 = time.perf_counter() - t0

print(f'=== M2: await secuencial ===')
for r in resultados_m2:
    print(f'  {r}')
print(f'Tiempo total M2: {t_m2:.2f}s  (esperado: {N_TAREAS * DURACION:.1f}s = N×T)')
print()

=== M2: await secuencial ===
  τ1: 1.00s
  τ2: 1.00s
  τ3: 1.01s
  τ4: 1.00s
  τ5: 1.00s
Tiempo total M2: 5.02s  (esperado: 5.0s = N×T)



In [9]:
# --- M4: asyncio.gather (esperas SÍ explotadas) ---
t0 = time.perf_counter()
resultados_m4 = await asyncio.gather(
    *[tarea_io(f'τ{i+1}', DURACION) for i in range(N_TAREAS)]
)
t_m4 = time.perf_counter() - t0

print(f'=== M4: asyncio.gather ===')
for r in resultados_m4:
    print(f'  {r}')
print(f'Tiempo total M4: {t_m4:.2f}s  (esperado: ~{DURACION:.1f}s = T_max)')
print()
print(f'Speedup M4/M2: {t_m2/t_m4:.1f}x')
print()
print(f'Conclusión: gather explota las esperas — exec(τⱼ) ∩ wait(τᵢ) ≠ ∅')
print(f'Las {N_TAREAS} tareas de {DURACION}s corren en ~{DURACION}s en lugar de {N_TAREAS*DURACION}s')

=== M4: asyncio.gather ===
  τ1: 1.00s
  τ2: 1.00s
  τ3: 1.00s
  τ4: 1.00s
  τ5: 1.00s
Tiempo total M4: 1.01s  (esperado: ~1.0s = T_max)

Speedup M4/M2: 5.0x

Conclusión: gather explota las esperas — exec(τⱼ) ∩ wait(τᵢ) ≠ ∅
Las 5 tareas de 1.0s corren en ~1.0s en lugar de 5.0s


## Sección 2: Traza del event loop con asyncio debug mode

asyncio tiene un modo de depuración que muestra advertencias cuando el event loop se bloquea.
Aquí vemos la diferencia entre `asyncio.sleep` y `time.sleep`.

In [ ]:
import asyncio
import time

# Habilitamos debug mode para ver bloqueos
loop = asyncio.get_event_loop()
loop.set_debug(True)

# Un umbral bajo para detectar bloqueos rápidamente
# (normalmente el umbral es 100ms)
loop.slow_callback_duration = 0.05  # 50ms

# Tarea bien escrita: libera el event loop
async def tarea_correcta(nombre: str):
    print(f'  {nombre}: inicio')
    await asyncio.sleep(0.3)   # wait(τ) — event loop libre
    print(f'  {nombre}: fin')

# Tarea mal escrita: BLOQUEA el event loop
async def tarea_bloqueante(nombre: str):
    print(f'  {nombre}: inicio')
    time.sleep(0.3)            # ← bloquea el hilo del OS entero
    print(f'  {nombre}: fin')

# ¿Qué diferencia ves en la salida?
print('=== gather con tareas CORRECTAS (asyncio.sleep) ===')
t0 = time.perf_counter()
await asyncio.gather(tarea_correcta('A'), tarea_correcta('B'), tarea_correcta('C'))
print(f'Tiempo: {time.perf_counter()-t0:.2f}s  (esperado: ~0.3s)\n')

print('=== gather con tareas BLOQUEANTES (time.sleep) ===')
t0 = time.perf_counter()
await asyncio.gather(tarea_bloqueante('X'), tarea_bloqueante('Y'), tarea_bloqueante('Z'))
print(f'Tiempo: {time.perf_counter()-t0:.2f}s  (esperado: ~0.9s — sin mejora)')
print()
print('Observa: con time.sleep, gather NO ayuda.')
print('time.sleep bloquea el event loop → ninguna otra coroutine puede avanzar.')


=== gather con tareas CORRECTAS (asyncio.sleep) ===
  A: inicio
  B: inicio
  C: inicio
  A: fin
  B: fin
  C: fin
Tiempo: 0.32s  (esperado: ~0.3s)

=== gather con tareas BLOQUEANTES (time.sleep) ===
  X: inicio
  X: fin
  Y: inicio
  Y: fin
  Z: inicio
  Z: fin
Tiempo: 0.91s  (esperado: ~0.9s — sin mejora)

Observa: con time.sleep, gather NO ayuda.
time.sleep bloquea el event loop → ninguna otra coroutine puede avanzar.


In [5]:
# Desactivar debug mode para el resto del notebook
loop.set_debug(False)

---

## Sección 3: Implementar M2 y M3 — por qué NO mejoran

Implementa los modelos M2 y M3 explícitamente y mide por qué no producen mejora sobre M1 para sus respectivos casos.

In [28]:
# TAREA 3.1 — M2: async con await secuencial (ya visto en Sección 1)
# Pregunta: ¿por qué M2 es idéntico a M1 en términos de tiempo?
# Responde con la definición formal: ¿qué condición de M4 falta en M2?

# TAREA 3.2 — M3: threading CPU-bound
# Implementa N tareas CPU-bound con threading y mide vs secuencial.
# ¿Coincide con la predicción del GIL (sin speedup, posible slowdown)?

def tarea_cpu_bound(n: int) -> int:
    """Tarea CPU-bound pura: wait(τᵢ) = ∅"""
    return sum(range(n))

N_CPU = 30_000_000
N_HILOS = 4

# --- Secuencial (M1) ---
t0 = time.perf_counter()
for _ in range(N_HILOS):
    tarea_cpu_bound(N_CPU)
t_secuencial = time.perf_counter() - t0


# --- Threading M3 ---
# TODO: implementa con threading.Thread y mide el tiempo
# Pista: usa la misma tarea_cpu_bound con N_HILOS hilos

t0 = time.perf_counter()
hilos = []
hilos = [threading.Thread(target=tarea_cpu_bound, args=(N_CPU,)) for _ in range(N_HILOS)]
for h in hilos: h.start()
for h in hilos: h.join()
t_threading = time.perf_counter() - t0




# TODO: imprime los tiempos y el speedup
# ¿Qué dice el resultado sobre M3 + GIL en Python?


print(f'M1 secuencial: {t_secuencial:.2f}s')
print(f'M3 Threading:   {(t_threading):.2f}s')
print(f'Speedup:     {t_secuencial/(t_threading):.2f}x  (esperado ~2x, real ~1x por el GIL)')

M1 secuencial: 0.92s
M3 Threading:   0.93s
Speedup:     0.98x  (esperado ~2x, real ~1x por el GIL)


## Sección 4: Race condition reproducible + fix con Lock

Las condiciones de carrera son consecuencia de la memoria compartida en concurrencia.
Reproduce el problema y aplica la solución con `threading.Lock`.

In [23]:
import threading
import time

# TAREA 4.1 — Reproduce la race condition
N_INCREMENTOS = 50_000
N_HILOS_RACE = 8
INTENTOS = 5  # repetir para que la no-determinación se manifieste

# Sin lock — resultado no determinista

def corrida_sin_lock() -> tuple[int, int]:
    contador_sin_lock = [0]

    def incrementar_sin_lock():
        for i in range(N_INCREMENTOS):
            # Todos los obstáculos: read-modify-write + yield entre pasos
            tmp = contador_sin_lock[0]
            if i % 50 == 0:
                time.sleep(0)  # cede CPU entre lectura y escritura
            tmp += 1
            contador_sin_lock[0] = tmp

    hilos = [threading.Thread(target=incrementar_sin_lock) for _ in range(N_HILOS_RACE)]
    for h in hilos:
        h.start()
    for h in hilos:
        h.join()

    esperado = N_INCREMENTOS * N_HILOS_RACE
    return esperado, contador_sin_lock[0]

for intento in range(1, INTENTOS + 1):
    esperado, obtenido = corrida_sin_lock()
    diff = esperado - obtenido
    print(f'Intento {intento}: esperado={esperado:,}, obtenido={obtenido:,}, perdidos={diff:,}')


# TAREA 4.2 — Fix con Lock
# El resultado debe ser siempre exactamente N_INCREMENTOS × N_HILOS_RACE

def corrida_con_lock() -> tuple[int, int]:
    contador_con_lock = [0]
    lock = threading.Lock()

    def incrementar_con_lock():
        for i in range(N_INCREMENTOS):
            # Mismos obstáculos que en sin lock, pero protegidos por sección crítica
            with lock:
                tmp = contador_con_lock[0]
                if i % 50 == 0:
                    time.sleep(0)  # incluso cediendo CPU, el lock evita interleaving peligroso
                tmp += 1
                contador_con_lock[0] = tmp

    hilos = [threading.Thread(target=incrementar_con_lock) for _ in range(N_HILOS_RACE)]
    for h in hilos:
        h.start()
    for h in hilos:
        h.join()

    esperado = N_INCREMENTOS * N_HILOS_RACE
    return esperado, contador_con_lock[0]

print('\nCon lock + mismos obstáculos (debe coincidir siempre):')
for intento in range(1, INTENTOS + 1):
    esperado, obtenido = corrida_con_lock()
    ok = 'OK' if obtenido == esperado else 'ERROR'
    print(f'Intento {intento}: esperado={esperado:,}, obtenido={obtenido:,} -> {ok}')

Intento 1: esperado=400,000, obtenido=50,200, perdidos=349,800
Intento 2: esperado=400,000, obtenido=50,050, perdidos=349,950
Intento 3: esperado=400,000, obtenido=50,050, perdidos=349,950
Intento 4: esperado=400,000, obtenido=50,050, perdidos=349,950
Intento 5: esperado=400,000, obtenido=50,000, perdidos=350,000

Con lock + mismos obstáculos (debe coincidir siempre):
Intento 1: esperado=400,000, obtenido=400,000 -> OK
Intento 2: esperado=400,000, obtenido=400,000 -> OK
Intento 3: esperado=400,000, obtenido=400,000 -> OK
Intento 4: esperado=400,000, obtenido=400,000 -> OK
Intento 5: esperado=400,000, obtenido=400,000 -> OK


## Sección 5: Chatbot v2 con asyncio — N usuarios concurrentes

Implementa el servidor chatbot v2 usando asyncio y mide su comportamiento con N usuarios.

In [24]:
import asyncio
import time
import random

# Operaciones I/O-bound del chatbot (simuladas)
async def consultar_bd(user_id: int) -> list:
    """wait(τᵢ): I/O a base de datos — ~50ms"""
    await asyncio.sleep(0.05)
    return [f'historial de usuario {user_id}']

async def llamar_llm(historial: list) -> str:
    """wait(τᵢ): I/O a API del LLM — 1–2s variable"""
    await asyncio.sleep(random.uniform(1.0, 2.0))
    return f'respuesta para: {historial[-1]}'

async def handle_request(user_id: int) -> dict:
    """Una petición completa del chatbot v2 (M4)"""
    # exec(τᵢ)
    t_inicio = time.perf_counter()

    # wait(τᵢ): BD
    historial = await consultar_bd(user_id)

    # wait(τᵢ): LLM
    respuesta = await llamar_llm(historial)

    # exec(τᵢ)
    latencia = time.perf_counter() - t_inicio
    return {'user': user_id, 'respuesta': respuesta, 'latencia': latencia}

# TAREA 5.1 — Servidor secuencial (chatbot v1 como baseline)
async def servidor_v1(n_usuarios: int):
    """M1: un usuario a la vez"""
    # DONE: servidor secuencial (for + await por usuario)
    resultados = []
    for i in range(n_usuarios):
        resultados.append(await handle_request(i))
    return resultados

# TAREA 5.2 — Servidor concurrente (chatbot v2)
async def servidor_v2(n_usuarios: int):
    """M4: todos los usuarios concurrentes con gather"""
    # DONE: servidor concurrente con asyncio.gather
    tareas = [handle_request(i) for i in range(n_usuarios)]
    return await asyncio.gather(*tareas)

# TAREA 5.3 — Compara v1 vs v2 con N=10 usuarios
# DONE: medir tiempos totales y latencias promedio
# DONE: calcular speedup v2/v1
# DONE: comparar latencias por usuario

N = 10
print(f'Comparando v1 vs v2 con {N} usuarios...')

# Medición v1
random.seed(42)
t0 = time.perf_counter()
res_v1 = await servidor_v1(N)
t_v1 = time.perf_counter() - t0
lat_prom_v1 = sum(r['latencia'] for r in res_v1) / N

# Medición v2
random.seed(42)
t0 = time.perf_counter()
res_v2 = await servidor_v2(N)
t_v2 = time.perf_counter() - t0
lat_prom_v2 = sum(r['latencia'] for r in res_v2) / N

speedup = t_v1 / t_v2

print('\n=== Resultados ===')
print(f'v1 secuencial:  tiempo total = {t_v1:.2f}s, latencia promedio = {lat_prom_v1:.2f}s')
print(f'v2 concurrente: tiempo total = {t_v2:.2f}s, latencia promedio = {lat_prom_v2:.2f}s')
print(f'Speedup v2/v1:  {speedup:.2f}x')

print('\nPrimeras 3 latencias por usuario:')
for i in range(min(3, N)):
    print(f"  user {i}: v1={res_v1[i]['latencia']:.2f}s | v2={res_v2[i]['latencia']:.2f}s")

print('\nConclusión:')
print('- v2 reduce fuertemente el tiempo total al solapar esperas I/O con gather.')
print('- La latencia individual sigue en el rango BD + LLM (~1-2s), pero el throughput total mejora mucho.')

Comparando v1 vs v2 con 10 usuarios...

=== Resultados ===
v1 secuencial:  tiempo total = 14.61s, latencia promedio = 1.46s
v2 concurrente: tiempo total = 1.96s, latencia promedio = 1.46s
Speedup v2/v1:  7.47x

Primeras 3 latencias por usuario:
  user 0: v1=1.70s | v2=1.70s
  user 1: v1=1.09s | v2=1.08s
  user 2: v1=1.34s | v2=1.33s

Conclusión:
- v2 reduce fuertemente el tiempo total al solapar esperas I/O con gather.
- La latencia individual sigue en el rango BD + LLM (~1-2s), pero el throughput total mejora mucho.
